# CardioSense — Phase 1 · Colab Setup

**Run this notebook first, in every new Colab session.** It prepares the runtime
for `01_clinical_training`, `02_ecg_training` and `03_xray_training`.

What it does:

1. Check the GPU
2. Mount Google Drive
3. Clone / update the repository
4. Install dependencies (Colab-specific list — it does **not** reinstall torch)
5. Set `CARDIOSENSE_DATA_ROOT`
6. Verify the environment
7. Download the datasets (each in its own optional cell)

> **Before you start:** `Runtime → Change runtime type → Hardware accelerator → GPU`.
> The clinical pipeline runs fine on CPU; the ECG and X-ray pipelines need the GPU.

## 1. Check the runtime

Confirm you actually got a GPU before doing anything expensive.

In [1]:
import subprocess, sys

print("Python:", sys.version.split()[0])
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout)
except Exception:
    print("nvidia-smi failed -> no GPU attached.")
    print("Fix: Runtime > Change runtime type > Hardware accelerator > GPU, then rerun.")

try:
    import torch
    print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
          "| CUDA:", torch.version.cuda)
except ImportError:
    print("torch not importable (unexpected on Colab)")

Python: 3.13.15
nvidia-smi failed -> no GPU attached.
Fix: Runtime > Change runtime type > Hardware accelerator > GPU, then rerun.
torch: 2.11.0+cpu | CUDA available: False | CUDA: None


## 2. Mount Google Drive

Drive holds the datasets and the model checkpoints, so a dropped runtime costs
you minutes rather than hours.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

# Edit this ONE line if you keep the project somewhere else in Drive.
DRIVE_PROJECT = Path("/content/drive/MyDrive/CardioSense")
DRIVE_DATA = DRIVE_PROJECT / "data"

for sub in ("clinical", "ecg", "xray"):
    (DRIVE_DATA / sub).mkdir(parents=True, exist_ok=True)

print("Drive project folder:", DRIVE_PROJECT)
print("Data root          :", DRIVE_DATA)

Mounted at /content/drive
Drive project folder: /content/drive/MyDrive/CardioSense
Data root          : /content/drive/MyDrive/CardioSense/data


## 3. Clone or update the repository

The repo lives on Colab's local disk (fast) while data and checkpoints live on
Drive (persistent). Set `REPO_URL` to your own fork before first use.

In [4]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/artrai73/CardioSense.git"  # <-- EDIT ME
REPO_DIR = Path("/content/CardioSense")

if REPO_DIR.exists():
    print("Repo already present; pulling latest...")
    print(subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                         capture_output=True, text=True).stdout)
else:
    print(subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)],
                         capture_output=True, text=True).stdout)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
print(sorted(p.name for p in REPO_DIR.iterdir() if not p.name.startswith(".")))


cwd: /content/CardioSense
['README.md', 'configs', 'data', 'docs', 'models', 'notebooks', 'pyproject.toml', 'requirements-colab.txt', 'requirements.txt', 'results', 'scripts', 'src', 'tests']


## 4. Install dependencies

We install `requirements-colab.txt`, **not** `requirements.txt`.

Colab ships a `torch` build matched to its GPU driver. `pip install torch` pulls a
generic wheel that frequently reports `CUDA available: False` afterwards, and the
only cure is a factory reset. The Colab requirements file therefore omits torch,
torchvision, and everything else Colab already provides.

In [5]:
!pip install -q -r requirements-colab.txt
!pip install -q -e .

# Verify torch survived the install untouched.
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 15.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cardiosense (pyproject.toml) ... done
torch: 2.11.0+cpu | CUDA available: False


**If a cell above asked you to restart the runtime:** do it
(`Runtime → Restart session`), then re-run from cell 2. You do not need to
reinstall — Colab keeps the installed packages across a restart.

## 5. Point CardioSense at your data

This is the only machine-specific setting in the whole project. Every other path
resolves automatically from the repository root.

In [9]:
import os
import sys
from pathlib import Path

# ============================================================
# 1. DEFINE PROJECT + DATA PATHS
# ============================================================

REPO_DIR = Path("/content/CardioSense")
SRC_DIR = REPO_DIR / "src"

DRIVE_PROJECT = Path(
    "/content/drive/MyDrive/CardioSense"
)

DRIVE_DATA = DRIVE_PROJECT / "data"


# ============================================================
# 2. VERIFY REPOSITORY
# ============================================================

print("=" * 70)
print("CARDIOSENSE PATH SETUP")
print("=" * 70)

print()
print("Repository:")
print(f"  {REPO_DIR}")

if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"CardioSense repository not found at:\n{REPO_DIR}\n\n"
        "Run the GitHub clone/setup cell first."
    )

print("✓ Repository exists")


# ============================================================
# 3. VERIFY SRC DIRECTORY
# ============================================================

print()
print("Source directory:")
print(f"  {SRC_DIR}")

if not SRC_DIR.exists():
    raise FileNotFoundError(
        f"CardioSense src directory not found at:\n{SRC_DIR}"
    )

print("✓ src directory exists")


# ============================================================
# 4. ADD SRC TO PYTHON PATH
# ============================================================

src_string = str(SRC_DIR)

if src_string not in sys.path:
    sys.path.insert(0, src_string)

print()
print("Python source path added:")
print(f"  {src_string}")


# ============================================================
# 5. VERIFY CARDIOSENSE PACKAGE
# ============================================================

print()
print("Testing CardioSense import...")

import cardiosense

print("✓ CardioSense import successful")
print(f"  {cardiosense.__file__}")


# ============================================================
# 6. CONFIGURE GOOGLE DRIVE DATA ROOT
# ============================================================

DRIVE_PROJECT.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_DATA.mkdir(
    parents=True,
    exist_ok=True
)

os.environ["CARDIOSENSE_DATA_ROOT"] = str(
    DRIVE_DATA
)

print()
print("Google Drive data root:")
print(f"  {DRIVE_DATA}")


# ============================================================
# 7. CREATE CARDIOSENSE PATHS
# ============================================================

from cardiosense.common.paths import ProjectPaths

# IMPORTANT:
# Create PATHS only AFTER CARDIOSENSE_DATA_ROOT is set.
PATHS = ProjectPaths.create()

PATHS.ensure_all()


# ============================================================
# 8. VERIFY PATHS.DATA
# ============================================================

print()
print("CardioSense PATHS.data:")
print(f"  {PATHS.data}")

if Path(PATHS.data).resolve() != DRIVE_DATA.resolve():

    raise RuntimeError(
        "PATHS.data is pointing to the wrong location!\n\n"
        f"Expected:\n  {DRIVE_DATA}\n\n"
        f"Got:\n  {PATHS.data}"
    )

print("✓ PATHS.data correctly points to Google Drive")


# ============================================================
# 9. FINAL STATUS
# ============================================================

print()
print("=" * 70)
print("CARDIOSENSE SETUP COMPLETE")
print("=" * 70)

print()
print("Code:")
print(f"  {REPO_DIR}")

print()
print("Python package:")
print(f"  {SRC_DIR / 'cardiosense'}")

print()
print("Data:")
print(f"  {PATHS.data}")

print()
print("Everything is configured correctly.")

CARDIOSENSE PATH SETUP

Repository:
  /content/CardioSense
✓ Repository exists

Source directory:
  /content/CardioSense/src
✓ src directory exists

Python source path added:
  /content/CardioSense/src

Testing CardioSense import...
✓ CardioSense import successful
  /content/CardioSense/src/cardiosense/__init__.py

Google Drive data root:
  /content/drive/MyDrive/CardioSense/data

CardioSense PATHS.data:
  /content/drive/MyDrive/CardioSense/data
✓ PATHS.data correctly points to Google Drive

CARDIOSENSE SETUP COMPLETE

Code:
  /content/CardioSense

Python package:
  /content/CardioSense/src/cardiosense

Data:
  /content/drive/MyDrive/CardioSense/data

Everything is configured correctly.


## 6. Verify the environment

One command, pass/warn/fail per check.

In [10]:
!python scripts/verify_setup.py --check-data

CARDIOSENSE RUNTIME ENVIRONMENT
  Python      : 3.13.15
  Platform    : Linux-6.6.122+-x86_64-with-glibc2.35
  CPU cores   : 2
  Colab       : True
  Git commit  : 5ce1382
  GPU         : none (cuda not available)
  Packages    :
      captum         0.9.0
      numpy          2.1.3
      pandas         2.2.3
      scipy          1.16.3
      shap           0.52.0
      sklearn        1.6.1
      torch          2.11.0+cpu
      torchvision    0.26.0+cpu
      wfdb           4.3.1
      xgboost        3.4.1

PROJECT PATHS
  root         /content/CardioSense
  data         /content/drive/MyDrive/CardioSense/data
  models       /content/CardioSense/models
  results      /content/CardioSense/results
  configs      /content/CardioSense/configs
  docs         /content/CardioSense/docs
  experiments  /content/CardioSense/results/experiments

1. Python
  [ ok ] Python 3.13

2. Packages
  [ ok ] numpy — 2.1.3
  [ ok ] pandas — 2.2.3
  [ ok ] sklearn — 1.6.1
  [ ok ] scipy — 1.16.3
  [ ok ] matp

In [11]:
# Same information from inside Python, plus the device the pipelines will use.
from cardiosense.common.env import get_device, print_environment
from cardiosense.common.seeding import set_seed

_ = print_environment()
device = get_device()
set_seed(42)

CARDIOSENSE RUNTIME ENVIRONMENT
  Python      : 3.13.15
  Platform    : Linux-6.6.122+-x86_64-with-glibc2.35
  CPU cores   : 2
  Colab       : True
  Git commit  : 5ce1382
  GPU         : none (cuda not available)
  Packages    :
      captum         0.9.0
      numpy          2.1.3
      pandas         2.2.3
      scipy          1.16.3
      shap           0.52.0
      sklearn        1.6.1
      torch          2.11.0+cpu
      torchvision    0.26.0+cpu
      wfdb           4.3.1
      xgboost        3.4.1
[env] device: cpu
[env] WARNING: no GPU detected. On Colab: Runtime > Change runtime type > Hardware accelerator > GPU.
[seed] seed=42 mode=seeded (cudnn.benchmark on)


42

## 7. Datasets

Three independent downloads. Run only the ones you need — each cell is safe to
re-run and skips work that is already done.

| Dataset | Size | Manual step | Time |
|---|---|---|---|
| UCI Heart Disease | < 100 KB | none | seconds |
| PTB-XL | ~1.7 GB zip | none (open access) | ~5-10 min |
| ChestX-ray14 | ~45 GB | Kaggle API token | ~30-45 min |

Full details: `docs/datasets.md`.

### 7a. Clinical — UCI Heart Disease (automatic)

In [ ]:
from pathlib import Path
import pandas as pd
from ucimlrepo import fetch_ucirepo

from cardiosense.common.config import load_config
from cardiosense.common.paths import PATHS

cfg = load_config("clinical")
cache = PATHS.data / "clinical" / Path(cfg.dataset.raw_cache_path).name

if cache.exists():
    df = pd.read_csv(cache)
    print(f"Loaded cached copy: {cache}")
else:
    repo = fetch_ucirepo(id=int(cfg.dataset.uci_repo_id))
    df = pd.concat([repo.data.features, repo.data.targets], axis=1)
    cache.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(cache, index=False)
    print(f"Downloaded and cached: {cache}")

print("shape:", df.shape)
print("columns:", list(df.columns))
print("\nmissing values per column:")
print(df.isna().sum()[lambda s: s > 0])
df.head()

### 7b. ECG — PTB-XL (open access, one wget)

Downloads to Colab local disk first, then moves into Drive. Roughly 1.7 GB
zipped. If you are tight on Drive space, delete `records500/` afterwards —
Phase 1 only reads `records100/`.

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from cardiosense.common.paths import PATHS

PTBXL_DIR = PATHS.data / "ecg" / "ptbxl"
URL = ("https://physionet.org/static/published-projects/ptb-xl/"
       "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip")

if (PTBXL_DIR / "ptbxl_database.csv").exists():
    print(f"PTB-XL already present at {PTBXL_DIR}")
else:
    print("Downloading PTB-XL (~1.7 GB)...")
    subprocess.run(["wget", "-q", "--show-progress", URL, "-O", "/content/ptbxl.zip"], check=True)
    print("Unzipping...")
    subprocess.run(["unzip", "-q", "/content/ptbxl.zip", "-d", "/content/ptbxl_extract"], check=True)

    extracted = next(Path("/content/ptbxl_extract").glob("ptb-xl-*"))
    PTBXL_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"Moving to {PTBXL_DIR} (this is the slow part - Drive I/O)...")
    shutil.move(str(extracted), str(PTBXL_DIR))
    os.remove("/content/ptbxl.zip")

print("\nContents:")
for item in sorted(PTBXL_DIR.iterdir())[:12]:
    print("  ", item.name)

In [ ]:
# Verify PTB-XL: metadata shape, fold structure, and one readable waveform.
import ast
import pandas as pd
import wfdb
from cardiosense.common.paths import PATHS

PTBXL_DIR = PATHS.data / "ecg" / "ptbxl"
db = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)

print("records          :", len(db))
print("unique patients  :", db.patient_id.nunique())
print("strat_fold values:", sorted(db.strat_fold.unique()))
print("diagnostic codes :", int(scp.diagnostic.fillna(0).sum()))
print("superclasses     :", sorted(scp[scp.diagnostic == 1].diagnostic_class.dropna().unique()))

first = db.iloc[0]
signal, meta = wfdb.rdsamp(str(PTBXL_DIR / first.filename_lr))
print(f"\nsample record shape: {signal.shape}  (samples x leads)")
print(f"sampling rate      : {meta['fs']} Hz")
print(f"leads              : {meta['sig_name']}")
print(f"scp_codes          : {ast.literal_eval(first.scp_codes)}")

### 7c. X-ray — NIH ChestX-ray14 (Kaggle)

Needs a Kaggle API token: **Kaggle → Settings → API → Create New Token**, which
downloads `kaggle.json`. Upload it with the cell below.

The raw archive is ~45 GB and will not fit in a free 15 GB Drive. Strategy:
download to Colab's local disk, build the filtered PA-Cardiomegaly subset once,
and copy only that subset (~1.5 GB) to Drive.

In [ ]:
# Upload kaggle.json, then install it where the CLI expects it.
import os
from pathlib import Path
from google.colab import files

if not Path("/root/.kaggle/kaggle.json").exists():
    print("Select your kaggle.json:")
    files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    os.replace("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle credentials ready.")

In [ ]:
# Download + flatten. ~30-45 min the first time. Safe to re-run.
import subprocess
from pathlib import Path
from cardiosense.common.paths import PATHS

RAW = Path("/content/nih_raw")
DEST = PATHS.data / "xray" / "nih"
DEST.mkdir(parents=True, exist_ok=True)
(DEST / "images").mkdir(exist_ok=True)

if any((DEST / "images").glob("*.png")):
    n = len(list((DEST / 'images').glob('*.png')))
    print(f"{n} images already at {DEST/'images'} - skipping download.")
else:
    RAW.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "nih-chest-xrays/data",
                    "-p", str(RAW), "--unzip"], check=True)

    # Kaggle ships images_001/images/ ... images_012/images/. Flatten them.
    subprocess.run(f"find {RAW} -path '*/images/*.png' -exec mv -t {DEST}/images {{}} +",
                   shell=True, check=True)
    subprocess.run(f"cp {RAW}/Data_Entry_2017*.csv {RAW}/*_list.txt {DEST}/",
                   shell=True, check=True)

    # The Kaggle mirror names it Data_Entry_2017.csv; the config expects the
    # NIH v2020 filename. Normalise so both work.
    legacy = DEST / "Data_Entry_2017.csv"
    if legacy.exists() and not (DEST / "Data_Entry_2017_v2020.csv").exists():
        legacy.rename(DEST / "Data_Entry_2017_v2020.csv")

print("\nContents of", DEST)
for item in sorted(DEST.iterdir()):
    print("  ", item.name)

In [ ]:
# Verify the X-ray metadata and look at the class balance we are about to face.
import pandas as pd
from cardiosense.common.config import load_config
from cardiosense.common.paths import PATHS

cfg = load_config("xray")
DEST = PATHS.data / "xray" / "nih"
meta_path = DEST / cfg.dataset.metadata_csv
if not meta_path.exists():
    meta_path = DEST / "Data_Entry_2017.csv"

meta = pd.read_csv(meta_path)
print("rows            :", len(meta))
print("unique patients :", meta[cfg.dataset.patient_column].nunique())
print("views           :", meta[cfg.dataset.view_column].value_counts().to_dict())

target = cfg.dataset.target_label
meta["is_target"] = meta[cfg.dataset.label_column].str.contains(target, regex=False)
pa = meta[meta[cfg.dataset.view_column] == cfg.dataset.filter_view]

print(f"\n{target} (all views): {int(meta.is_target.sum())} "
      f"({100 * meta.is_target.mean():.2f}%)")
print(f"{target} (PA only)  : {int(pa.is_target.sum())} of {len(pa)} "
      f"({100 * pa.is_target.mean():.2f}%)")
print("\nThis is why PR-AUC, not accuracy, is the headline metric for this pipeline.")

## 8. Ready

If `verify_setup.py` reported no failures and the dataset cells you need have
completed, continue to:

- `01_clinical_training.ipynb`
- `02_ecg_training.ipynb`
- `03_xray_training.ipynb`

**Every session from now on:** re-run sections 2–6 of this notebook. The Drive
mount, the `CARDIOSENSE_DATA_ROOT` env var and the pip installs do not survive a
runtime reset. The datasets on Drive do.

In [ ]:
# Optional: run the CPU-only smoke tests. Fast, needs no datasets.
!python -m pytest tests/ -q